# BERT + Global Context for NER


## 1. Cài đặt thư viện

In [1]:
!pip install transformers datasets evaluate seqeval torch accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


## 2. Imports & Config

In [2]:
import os, requests, torch, torch.nn as nn, numpy as np
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Cell 3: Config
MODEL_NAME = 'bert-base-cased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10
LR = 3e-5
DROPOUT = 0.1

TRAIN_FILE = 'train.conll'
VALID_FILE = 'valid.conll'
TEST_FILE  = 'test.conll'

Using device: cuda
GPU: Tesla T4


## 3. Tải dữ liệu

In [3]:
base_url = "https://raw.githubusercontent.com/kodomotachi/NER/main/data/processed"

for fname in [TRAIN_FILE, VALID_FILE, TEST_FILE]:
    r = requests.get(f"{base_url}/{fname}")
    r.raise_for_status()
    with open(fname, 'wb') as f:
        f.write(r.content)
    print(f"Đã tải: {fname} ({len(r.content):,} bytes)")

Đã tải: train.conll (1,807,214 bytes)
Đã tải: valid.conll (222,414 bytes)
Đã tải: test.conll (224,520 bytes)


## 4. Đọc CoNLL & Build Labels

In [4]:
# Cell 4: Read CoNLL files
def read_conll(filepath):
    sentences, tokens, tags = [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split('\t')
                if len(parts) == 2:
                    tokens.append(parts[0])
                    tags.append(parts[1])
    if tokens:
        sentences.append((tokens, tags))
    return sentences

train_data = read_conll(TRAIN_FILE)
valid_data = read_conll(VALID_FILE)
test_data  = read_conll(TEST_FILE)

print(f'Train: {len(train_data)} docs')
print(f'Valid: {len(valid_data)} docs')
print(f'Test:  {len(test_data)} docs')

Train: 2527 docs
Valid: 316 docs
Test:  316 docs


In [5]:
# Cell 5: Build label mapping
all_tags = set()
for tokens, tags in train_data + valid_data + test_data:
    all_tags.update(tags)

label_list = sorted(all_tags)
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
num_labels = len(label_list)

print(f'Labels ({num_labels}): {label_list}')

Labels (23): ['B-BUY_G', 'B-BUY_N', 'B-GSTL', 'B-GT_AMT', 'B-GT_AMTL', 'B-INV_DL', 'B-INV_DT', 'B-INV_L', 'B-INV_NO', 'B-SUPP_G', 'B-SUPP_N', 'I-BUY_G', 'I-BUY_N', 'I-GSTL', 'I-GT_AMT', 'I-GT_AMTL', 'I-INV_DL', 'I-INV_DT', 'I-INV_L', 'I-INV_NO', 'I-SUPP_G', 'I-SUPP_N', 'O']


## 5. Dataset Class

In [6]:
# Cell 6: Dataset class
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NERDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, tags = self.data[idx]
        tokens = tokens[:self.max_len - 2]
        tags = tags[:self.max_len - 2]

        input_ids = [self.tokenizer.cls_token_id]
        label_ids = [self.label2id['O']]

        for word, tag in zip(tokens, tags):
            word_tokens = self.tokenizer.encode(word, add_special_tokens=False)
            if not word_tokens:
                continue
            input_ids.extend(word_tokens)
            label_ids.append(self.label2id[tag])
            for _ in range(1, len(word_tokens)):
                if tag.startswith('B-'):
                    label_ids.append(self.label2id.get('I-' + tag[2:], self.label2id[tag]))
                else:
                    label_ids.append(self.label2id[tag])

        input_ids = input_ids[:self.max_len - 1]
        label_ids = label_ids[:self.max_len - 1]

        input_ids.append(self.tokenizer.sep_token_id)
        label_ids.append(self.label2id['O'])

        seq_len = len(input_ids)
        pad_len = self.max_len - seq_len
        attention_mask = [1] * seq_len + [0] * pad_len
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        label_ids += [self.label2id['O']] * pad_len

        return {
            'input_ids':      torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels':         torch.tensor(label_ids, dtype=torch.long),
            'seq_len':        torch.tensor(seq_len, dtype=torch.long),
        }

train_ds = NERDataset(train_data, tokenizer, label2id, MAX_LEN)
valid_ds = NERDataset(valid_data, tokenizer, label2id, MAX_LEN)
test_ds  = NERDataset(test_data,  tokenizer, label2id, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE*2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2)

print(f'Batches - train: {len(train_loader)}  valid: {len(valid_loader)}  test: {len(test_loader)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Batches - train: 158  valid: 10  test: 10


## 6. Mô hình BERT + Global Context

```
Input → BERT → Token Embeddings (H)
                    ↓
              ┌─────┴─────┐
              │            │
         Token H_i    Global Context g
              │         (attention pooling)
              └─────┬─────┘
                    │
              Concat [H_i; g]
                    │
              Projection → Classifier → Labels
```

In [7]:
# Cell 7: BERT + Global Context Model
class GlobalContextAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 4),
            nn.Tanh(),
            nn.Linear(hidden_size // 4, 1)
        )

    def forward(self, hidden_states, attention_mask):
        attn_scores = self.attention(hidden_states).squeeze(-1)
        attn_scores = attn_scores.masked_fill(attention_mask == 0, float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        global_ctx = torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)
        return global_ctx, attn_weights


class BERT_GlobalContext(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.global_context = GlobalContextAttention(hidden_size)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeds = outputs.last_hidden_state

        global_ctx, attn_weights = self.global_context(token_embeds, attention_mask)
        global_ctx_expanded = global_ctx.unsqueeze(1).expand_as(token_embeds)

        fused = torch.cat([token_embeds, global_ctx_expanded], dim=-1)
        fused = self.fusion(fused)
        logits = self.classifier(fused)

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            return loss, logits
        return logits

model = BERT_GlobalContext(MODEL_NAME, num_labels, DROPOUT).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model parameters: 109,657,752


## 7. Optimizer & Scheduler

In [8]:
# Cell 8: Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

## 8. Evaluation Function

In [9]:
# Cell 9: Evaluation function
def evaluate(model, dataloader, id2label):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            seq_lens = batch['seq_len']

            logits = model(input_ids, attention_mask)
            preds = logits.argmax(-1)

            for i in range(len(preds)):
                slen = seq_lens[i].item()
                true_tags = [id2label[labels[i][j].item()] for j in range(1, slen - 1)]
                pred_tags = [id2label[preds[i][j].item()] for j in range(1, slen - 1)]
                y_true.append(true_tags)
                y_pred.append(pred_tags)

    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    f = f1_score(y_true, y_pred)
    return p, r, f, y_true, y_pred

## 9. Training Loop

In [10]:
# Cell 10: Training loop
best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        loss, logits = model(input_ids, attention_mask, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f'  Epoch {epoch} Step {step+1}/{len(train_loader)} Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    p, r, f1, _, _ = evaluate(model, valid_loader, id2label)
    print(f'Epoch {epoch}/{EPOCHS} - Loss: {avg_loss:.4f} | Val P: {p:.4f} R: {r:.4f} F1: {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best_bert_global_context.pt')
        print(f'  -> Saved best model (F1: {f1:.4f})')

print(f'\nTraining complete. Best Val F1: {best_f1:.4f}')

  Epoch 1 Step 50/158 Loss: 0.3071
  Epoch 1 Step 100/158 Loss: 0.1376
  Epoch 1 Step 150/158 Loss: 0.0195
Epoch 1/10 - Loss: 0.5818 | Val P: 0.9201 R: 0.9597 F1: 0.9395
  -> Saved best model (F1: 0.9395)
  Epoch 2 Step 50/158 Loss: 0.0063
  Epoch 2 Step 100/158 Loss: 0.0383
  Epoch 2 Step 150/158 Loss: 0.0029
Epoch 2/10 - Loss: 0.0177 | Val P: 0.9766 R: 0.9847 F1: 0.9806
  -> Saved best model (F1: 0.9806)
  Epoch 3 Step 50/158 Loss: 0.0083
  Epoch 3 Step 100/158 Loss: 0.0107
  Epoch 3 Step 150/158 Loss: 0.0022
Epoch 3/10 - Loss: 0.0088 | Val P: 0.9821 R: 0.9917 F1: 0.9869
  -> Saved best model (F1: 0.9869)
  Epoch 4 Step 50/158 Loss: 0.0023
  Epoch 4 Step 100/158 Loss: 0.0025
  Epoch 4 Step 150/158 Loss: 0.0018
Epoch 4/10 - Loss: 0.0055 | Val P: 0.9834 R: 0.9889 F1: 0.9861
  Epoch 5 Step 50/158 Loss: 0.0017
  Epoch 5 Step 100/158 Loss: 0.0021
  Epoch 5 Step 150/158 Loss: 0.0018
Epoch 5/10 - Loss: 0.0048 | Val P: 0.9889 R: 0.9931 F1: 0.9910
  -> Saved best model (F1: 0.9910)
  Epoch 6 

## 10. Test Evaluation

In [11]:
# Cell 11: Test evaluation
model.load_state_dict(torch.load('best_bert_global_context.pt'))
p, r, f1, y_true, y_pred = evaluate(model, test_loader, id2label)

print('='*60)
print('TEST RESULTS - BERT + Global Context')
print('='*60)
print(f'Precision: {p:.4f}')
print(f'Recall:    {r:.4f}')
print(f'F1-Score:  {f1:.4f}')
print('='*60)
print()
print(classification_report(y_true, y_pred))

TEST RESULTS - BERT + Global Context
Precision: 0.9730
Recall:    0.9863
F1-Score:  0.9796

              precision    recall  f1-score   support

       BUY_G       0.95      0.97      0.96        62
       BUY_N       0.93      0.96      0.94        68
        GSTL       0.99      1.00      1.00       119
      GT_AMT       0.96      1.00      0.98        22
     GT_AMTL       0.92      0.96      0.94        23
      INV_DL       0.97      0.97      0.97        75
      INV_DT       0.97      0.99      0.98        74
       INV_L       0.97      0.99      0.98        75
      INV_NO       1.00      1.00      1.00        75
      SUPP_G       0.98      1.00      0.99        57
      SUPP_N       0.99      1.00      0.99        81

   micro avg       0.97      0.99      0.98       731
   macro avg       0.97      0.98      0.98       731
weighted avg       0.97      0.99      0.98       731



## 11. Save Model

In [12]:
# Cell 12: Save model for deployment
import json

os.makedirs('saved_model', exist_ok=True)
torch.save(model.state_dict(), 'saved_model/model.pt')
with open('saved_model/labels.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, f)
tokenizer.save_pretrained('saved_model')
print('Model saved to saved_model/')

Model saved to saved_model/
